# v0.28.0 — Query cache

## 1. Intro

Since v0.28.0 a QuerySet can serve its reads from a **client-side cache**: with `cache(ttl)`, the rows a read
returned stay in this process's memory, and the same read is answered from there, without a request, until its TTL
runs out or an ORM write changes what it reads. Invalidation is built in: every ORM write removes the cached reads it
can change, with nothing to set up. `QueryCache` holds the full SurrealDB-ORM's settings and helpers:
`configure()`, `make_key()`, `get()`, `set()`, `invalidate()`, `clear()` and `stats()`.

The cache lives in the ORM, not in SurrealDB: it behaves the same on SurrealDB 2.x, 3.x and the embedded engine.
Reads inside a transaction are never cached. This notebook runs top to bottom on either line; only the cell that
prints the server line and the transaction section (§10) differ.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


In [2]:
client = await SurrealDBConnectionManager.get_client()
version = str(await client.version())  # "surrealdb-<major>.<minor>.<patch>"
line = int(version.removeprefix("surrealdb-").split(".")[0])
print(f"SurrealDB {line}.x")

SurrealDB 3.x


## 3. Models and data

Five products. The notebook starts by dropping its tables, so it can be re-executed in place (on SurrealDB 3.x
removing a missing table raises, hence the suppressed errors), then empties the cache: `QueryCache.clear()` also
resets the `hits` and `misses` counters.

`show()` runs one read and prints its result and **where it came from**, told by `QueryCache.stats()`: a hit was
answered from the cache, a miss went to the database (and stored its rows), and a read the cache does not count
went to the database without it.

In [3]:
from collections.abc import Awaitable, Callable
from typing import Any

from surreal_orm_lite import BaseSurrealModel, QueryCache


class Product(BaseSurrealModel):
    id: str | None = None
    name: str
    category: str
    price: int


DEMO_TABLES = ["Product", "Review"]


async def reset() -> None:
    """Drop every table of this notebook, whether or not it exists."""
    for table in DEMO_TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE {table};", {})


async def show(label: str, read: Awaitable[Any], view: Callable[[Any], Any] = lambda result: result) -> None:
    """Run one read, print where it came from (QueryCache.stats() counts hits and misses) and its result."""
    before = QueryCache.stats()
    result = await read
    after = QueryCache.stats()
    if after["hits"] > before["hits"]:
        source = "cache (hit)"
    elif after["misses"] > before["misses"]:
        source = "database (miss)"
    else:
        source = "database (not counted)"
    print(f"{label:<42} {source:<23} {view(result)}")


def names(products: list[Product]) -> list[str]:
    """The names of the products, in the order read."""
    return [product.name for product in products]


def prices(products: list[Product]) -> list[tuple[str, int]]:
    """(name, price) of each product, in the order read."""
    return [(product.name, product.price) for product in products]


await reset()

for key, name, category, price in [
    ("p1", "Dune", "book", 12),
    ("p2", "Lamp", "home", 30),
    ("p3", "Emma", "book", 9),
    ("p4", "Chair", "home", 45),
    ("p5", "Kettle", "kitchen", 25),
]:
    await Product(id=key, name=name, category=category, price=price).save()

QueryCache.clear()
print("products:", prices(await Product.objects().order_by("name").exec()))
print("cache:   ", QueryCache.stats())

products: [('Chair', 45), ('Dune', 12), ('Emma', 9), ('Kettle', 25), ('Lamp', 30)]
cache:    {'entries': 0, 'tables': 0, 'max_size': 1000, 'default_ttl': 60.0, 'enabled': True, 'hits': 0, 'misses': 0}


## 4. A cached read, twice

`cache(ttl=30)` lets this read accept a result up to 30 seconds old. The first read goes to the database and stores
its rows; the second sends **the same statement with the same variables**, on the same connection and identity, so
it is answered from memory without a request: one miss, then one hit.

Rows are cached, not instances: each hit builds new instances from a copy, so changing a result never changes the
cache.

In [4]:
def books():
    """A new queryset each time: QuerySet methods change the queryset they are called on."""
    return Product.objects().filter(category="book").order_by("name").cache(ttl=30)


await show("books, 1st read:", books().exec(), names)
await show("books, 2nd read:", books().exec(), names)
stats = QueryCache.stats()
print("hits:", stats["hits"], "| misses:", stats["misses"])

result = await books().exec()
result[0].name = "Changed locally"
await show("books, after changing a result:", books().exec(), names)

books, 1st read:                           database (miss)         ['Dune', 'Emma']
books, 2nd read:                           cache (hit)             ['Dune', 'Emma']
hits: 1 | misses: 1
books, after changing a result:            cache (hit)             ['Dune', 'Emma']


## 5. Every read is cached

Not only `exec()`: `first()`, `get()` by filters or by id, `all()`, `count()`, `sum()`, `avg()`, `min()`, `max()`,
`exists()`, `values(…).annotate(…)` and `annotate(GeoDistance)` too. `cache()` without a TTL takes `QueryCache`'s
default when the read runs (60 seconds unless configured). The first round below is three misses, the second three
hits.

In [5]:
from surreal_orm_lite import Count


def by_category(rows: list[dict[str, Any]]) -> list[tuple[str, int]]:
    """(category, n) of each group, sorted."""
    return sorted((row["category"], row["n"]) for row in rows)


for round_number in (1, 2):
    print(f"round {round_number}")
    await show("  count():", Product.objects().cache().count())
    await show('  get("p1"):', Product.objects().cache().get("p1"), lambda product: product.name)
    await show(
        '  values("category").annotate(n=Count()):',
        Product.objects().values("category").annotate(n=Count()).cache().exec(),
        by_category,
    )

stats = QueryCache.stats()
print("hits:", stats["hits"], "| misses:", stats["misses"], "| entries:", stats["entries"])

round 1
  count():                                 database (miss)         5
  get("p1"):                               database (miss)         Dune
  values("category").annotate(n=Count()):  database (miss)         [('book', 2), ('home', 2), ('kitchen', 1)]
round 2
  count():                                 cache (hit)             5
  get("p1"):                               cache (hit)             Dune
  values("category").annotate(n=Count()):  cache (hit)             [('book', 2), ('home', 2), ('kitchen', 1)]
hits: 6 | misses: 4 | entries: 4


## 6. Invalidation: every ORM write

An ORM write removes the cached reads of the tables it writes, before its `post_*` signals fire. After `save()` of
a new book, the books read of §4 is a miss again, and Atlas is in it. The same goes for `bulk_update()` and every
other write (`update()`, `merge()`, `upsert()`, `patch()`, `delete()`, `bulk_create()`, `bulk_delete()`,
`QuerySet.patch()`, `delete_table()`, the atomic operations, relations…).

In [6]:
await show("books:", books().exec(), names)
await Product(id="p6", name="Atlas", category="book", price=40).save()
await show("books, after save() of Atlas:", books().exec(), names)


def home():
    return Product.objects().filter(category="home").order_by("name").cache()


await show("home:", home().exec(), prices)
await show("home, again:", home().exec(), prices)
updated = await Product.objects().filter(category="home").bulk_update(price=20)
print("bulk_update(price=20):", updated, "products updated")
await show("home, after bulk_update():", home().exec(), prices)

books:                                     cache (hit)             ['Dune', 'Emma']
books, after save() of Atlas:              database (miss)         ['Atlas', 'Dune', 'Emma']
home:                                      database (miss)         [('Chair', 45), ('Lamp', 30)]
home, again:                               cache (hit)             [('Chair', 45), ('Lamp', 30)]
bulk_update(price=20): 2 products updated
home, after bulk_update():                 database (miss)         [('Chair', 20), ('Lamp', 20)]


## 7. Reads through links

A review's `product` is a record link. Filtering reviews on `product.name` reads the **Product** table through the
link, though the statement reads `FROM Review`. The cache cannot tell which tables such a read depends on, so a read
that follows links (`fetch()`, a dotted path in a filter or an ordering, a `select()` item that is not a plain
field) is removed by **any** ORM write: here `merge()` on the product, which never touches `Review`, refreshes it.

In [7]:
from surrealdb import RecordID


class Review(BaseSurrealModel):
    id: str | None = None
    product: RecordID  # a record link to a Product
    stars: int


for key, product, stars in [("r1", "p1", 5), ("r2", "p2", 4), ("r3", "p3", 3)]:
    await Review(id=key, product=RecordID(Product.get_table_name(), product), stars=stars).save()


def dune_reviews():
    return Review.objects().filter(**{"product.name": "Dune"}).cache()


await show('reviews of "Dune":', dune_reviews().count())
await show('reviews of "Dune", again:', dune_reviews().count())

dune = await Product.objects().get("p1")
await dune.merge(name="Dune Messiah")  # a write to Product only
await show('reviews of "Dune", after merge():', dune_reviews().count())

reviews of "Dune":                         database (miss)         1
reviews of "Dune", again:                  cache (hit)             1
reviews of "Dune", after merge():          database (miss)         0


## 8. What the cache cannot see

The cache only sees the ORM's writes. Raw SurrealQL (here an `UPDATE` sent through the SDK client), writes by
another process or client, and what the server derives on its own (`DEFINE EVENT`, views, `REFERENCE … ON DELETE`)
leave the cached reads as they were: the TTL bounds how long, and `QueryCache.invalidate(Model)` removes a table's
entries (and every read that follows links) at once. It returns how many entries it removed: here the `get()`
above and the read of §7, which follows links.

In [8]:
def price(product: Product) -> int:
    return product.price


await show('get("p5"), price:', Product.objects().cache().get("p5"), price)
await client.query("UPDATE Product:p5 SET price = 28;", {})  # raw SurrealQL: the cache is not told
await show('get("p5"), price, after a raw UPDATE:', Product.objects().cache().get("p5"), price)

removed = QueryCache.invalidate(Product)
print("QueryCache.invalidate(Product) removed", removed, "entries")
await show('get("p5"), price, after invalidate():', Product.objects().cache().get("p5"), price)

get("p5"), price:                          database (miss)         25
get("p5"), price, after a raw UPDATE:      cache (hit)             25
QueryCache.invalidate(Product) removed 2 entries
get("p5"), price, after invalidate():      database (miss)         28


## 9. Time to live

`cache(ttl)` takes seconds (an `int` or a `float`) or a `timedelta`: it is the **oldest result this read accepts**.
Here half a second: the second read is a hit, the one 0.6 s later a miss. A TTL of `0` means never cached: both
reads go to the database.

In [9]:
import asyncio


def kitchen(ttl: float):
    return Product.objects().filter(category="kitchen").cache(ttl=ttl)


await show("kitchen count(), ttl=0.5:", kitchen(0.5).count())
await show("kitchen count(), ttl=0.5, again:", kitchen(0.5).count())
await asyncio.sleep(0.6)
await show("kitchen count(), ttl=0.5, 0.6 s later:", kitchen(0.5).count())

await show("kitchen count(), ttl=0:", kitchen(0).count())
await show("kitchen count(), ttl=0, again:", kitchen(0).count())

kitchen count(), ttl=0.5:                  database (miss)         1
kitchen count(), ttl=0.5, again:           cache (hit)             1


kitchen count(), ttl=0.5, 0.6 s later:     database (miss)         1
kitchen count(), ttl=0:                    database (miss)         1
kitchen count(), ttl=0, again:             database (miss)         1


## 10. Transactions

Reads inside a transaction (`objects(tx=)`) are never cached and never counted: on SurrealDB 3.x they may see the
transaction's uncommitted writes. The tables a transaction writes are invalidated **at commit**; until then, reads
outside the transaction see the committed state, from the cache.

On SurrealDB 2.x (and over HTTP) a transaction buffers its writes and reads inside it raise, with or without
`cache()`: the notebook probes the server and only writes there.

In [10]:
async def native_tx_supported() -> bool:
    """True if the server exposes the SDK's native transaction RPC (SurrealDB 3.x)."""
    try:
        txn = await client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await client.cancel(txn)
    return True


interactive = await native_tx_supported()
print("Interactive transactions available (SurrealDB 3.x):", interactive)

Interactive transactions available (SurrealDB 3.x): True


In [11]:
def total():
    return Product.objects().cache().count()


await show("count(), before the transaction:", total())
async with SurrealDBConnectionManager.transaction() as tx:
    await Product(id="p7", name="Vase", category="home", price=15).save(tx=tx)
    if interactive:
        await show("count() inside, objects(tx=tx).cache():", Product.objects(tx=tx).cache().count())
        await show("count() outside, during the transaction:", total())
    else:
        print("SurrealDB 2.x: the write is buffered until the commit; a read inside the transaction would raise.")
await show("count(), after the commit:", total())

count(), before the transaction:           database (miss)         6
count() inside, objects(tx=tx).cache():    database (not counted)  7
count() outside, during the transaction:   cache (hit)             6
count(), after the commit:                 database (miss)         7


## 11. What `cache()` refuses

`query()` runs your own SurrealQL, which the ORM never caches: with `cache()` it raises `SurrealDbError`. `live()`,
`watch()` and `changes()` stream, so they name `cache()` among the clauses their usual error refuses; `as_of()`
refuses it too (a time-travel read is never cached).

In [12]:
from surreal_orm_lite import SurrealDbError

try:
    await Product.objects().cache().query("SELECT * FROM Product")
except SurrealDbError as error:
    print(f"query():   {type(error).__name__}: {error}")

try:
    Product.objects().cache().changes()
except SurrealDbError as error:
    print(f"changes(): {type(error).__name__}: {error}")

query():   SurrealDbError: query() runs your own SurrealQL and is never cached: drop cache(), or cache its result yourself with QueryCache.make_key(), get() and set()
changes(): SurrealDbError: A change feed cannot honour cache(): SurrealDB's SHOW CHANGES reads the whole table's log, with no WHERE, projection, ordering or paging, outside any transaction. Call changes() on Model.objects() and filter the events.


To cache the result of your own SurrealQL, build a key with `make_key()` (from the statement, its variables as the
SDK sends them, and the table) and use `get()` and `set()`. The entry is tagged with the table, so an ORM write to
it removes the entry. `get()` counts a hit or a miss like a `cache()` read.

In [13]:
async def total_price() -> int:
    statement = "SELECT math::sum(price) AS total FROM Product GROUP ALL"
    key = QueryCache.make_key(statement, {}, "Product")
    rows = QueryCache.get(key)  # a copy, or None: missing, expired, or the cache is disabled
    if rows is None:
        rows = await Product.raw_query(statement)
        QueryCache.set(key, rows, "Product", ttl=60)
    return rows[0]["total"]


await show("total_price():", total_price())
await show("total_price(), again:", total_price())

total_price():                             database (miss)         144
total_price(), again:                      cache (hit)             144


## 12. Configuration

`QueryCache.configure(default_ttl=60, max_size=1000, enabled=True)` has the full SurrealDB-ORM's three settings, and
**every call sets all three**: an argument left out takes its default. `configure()` never clears the cache.

`enabled=False` **hides** the entries without removing them: `stats()` still counts them, a `cache()` read goes to
the database and nothing is counted (calling `cache()` logs a warning, printed here with the outputs).

In [14]:
import logging
import sys

# Print the ORM's warnings with the outputs of this notebook.
logging.basicConfig(stream=sys.stdout, format="%(levelname)s %(name)s: %(message)s")


def report(label: str) -> None:
    """Print the settings and counters of QueryCache.stats() after a step."""
    stats = QueryCache.stats()
    print(f"{label:<28}", {name: stats[name] for name in ("entries", "enabled", "default_ttl", "hits", "misses")})


def books_count():
    return Product.objects().filter(category="book").cache().count()


await show("books count():", books_count())
report("before:")
QueryCache.configure(enabled=False)
report("configure(enabled=False):")
await show("books count(), disabled:", books_count())
report("after the disabled read:")

books count():                             database (miss)         3
before:                      {'entries': 3, 'enabled': True, 'default_ttl': 60.0, 'hits': 13, 'misses': 19}
configure(enabled=False):    {'entries': 3, 'enabled': False, 'default_ttl': 60.0, 'hits': 13, 'misses': 19}
WARNING surreal_orm_lite.query_set: QueryCache is disabled — .cache() has no effect


books count(), disabled:                   database (not counted)  3
after the disabled read:     {'entries': 3, 'enabled': False, 'default_ttl': 60.0, 'hits': 13, 'misses': 19}


`configure(default_ttl=120)` sets `enabled` back to its default, `True`: the entries still valid are served again,
so the same read is a hit. `QueryCache.clear()` is the way to drop the rows from memory; it also resets `hits` and
`misses`.

In [15]:
QueryCache.configure(default_ttl=120)
report("configure(default_ttl=120):")
await show("books count(), enabled again:", books_count())

QueryCache.clear()
report("clear():")

configure(default_ttl=120):  {'entries': 3, 'enabled': True, 'default_ttl': 120.0, 'hits': 13, 'misses': 19}
books count(), enabled again:              cache (hit)             3
clear():                     {'entries': 0, 'enabled': True, 'default_ttl': 120.0, 'hits': 0, 'misses': 0}


## 13. Cleanup

Drop the demo tables and close the connection. Closing a connection also drops the cached results read through it.

In [16]:
await show("count():", Product.objects().cache().count())
await reset()
print("entries before close_connection():", QueryCache.stats()["entries"])
await SurrealDBConnectionManager.close_connection()
print("entries after close_connection():", QueryCache.stats()["entries"])
print("Done")

count():                                   database (miss)         7
entries before close_connection(): 1
entries after close_connection(): 0
Done
